In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Master_Thesis
!pwd
!pip install music21

In [ ]:
#input_folder = "/content/drive/MyDrive/Master_Thesis/data/all_data_augmented_text"
#output_folder = "/content/drive/MyDrive/Master_Thesis/result_test_musescore"

In [ ]:
import os
from itertools import groupby
from music21 import stream, chord, meter, tempo, instrument, clef, layout

def read_chords_from_txt(path, min_pitch=21, max_pitch=108):
    """
    Reads lines like '74 70 65 58' and returns a list of 4-int lists.
    Clamps pitches to [min_pitch, max_pitch].
    """
    chords = []
    with open(path, 'r') as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            parts = ln.split()
            if len(parts) != 4:
                continue
            try:
                pitches = [int(x) for x in parts]
            except ValueError:
                continue
            pitches = [max(min_pitch, min(max_pitch, p)) for p in pitches]
            chords.append(pitches)
    return chords

def compress_identical_runs(chords_4ints, equal_mode="exact"):
    """
    Compress consecutive identical chords into (pitches, run_len).
    equal_mode:
      - "exact": identical if all 4 ints match in the same order
      - "pitchset": identical if the multiset of pitches matches (order-insensitive)
    """
    def key_fn(v):
        return tuple(v) if equal_mode == "exact" else tuple(sorted(v))

    compressed = []
    for k, grp in groupby(chords_4ints, key=key_fn):
        grp_list = list(grp)
        # Keep the original pitches from the first item (preserves S/A/T/B order)
        first = grp_list[0]
        compressed.append((first, len(grp_list)))
    return compressed

def chorale_to_grand_staff_musicxml(
    chords_4ints,
    out_musicxml_path,
    out_midi_path,
    time_signature="4/4",
    tempo_qpm=84,
    sixteenth_qL=0.25,
    assume_order_SATB=True,
    merge_identical_runs=True,
    equal_mode="exact",  # "exact" or "pitchset"
):
    """
    Converts 4-int chords into a two-staff (treble/bass) score.
    If merge_identical_runs=True, consecutive identical chords are merged into longer durations.
    Longer durations that cross barlines will be automatically tied after makeMeasures/makeNotation.
    """
    # Optionally compress identical runs before building the score
    items = compress_identical_runs(chords_4ints, equal_mode) if merge_identical_runs \
            else [(p, 1) for p in chords_4ints]

    sc = stream.Score()
    sc.append(layout.PageLayout())

    treble = stream.Part()
    treble.partName = "Treble (S/A)"
    treble.insert(0, instrument.Piano())
    treble.insert(0, clef.TrebleClef())
    treble.insert(0, meter.TimeSignature(time_signature))
    treble.insert(0, tempo.MetronomeMark(number=tempo_qpm))

    bass = stream.Part()
    bass.partName = "Bass (T/B)"
    bass.insert(0, instrument.Piano())
    bass.insert(0, clef.BassClef())
    bass.insert(0, meter.TimeSignature(time_signature))
    bass.insert(0, tempo.MetronomeMark(number=tempo_qpm))

    for pitches, run_len in items:
        duration_qL = sixteenth_qL * run_len  # e.g., 2 identical 16ths -> 8th, 4 -> quarter, etc.

        if assume_order_SATB:
            s, a, t, b = pitches
            treble_notes = [s, a]
            bass_notes  = [t, b]
        else:
            sorted_p = sorted(pitches)
            bass_notes  = sorted_p[:2]
            treble_notes = sorted_p[2:]

        treble_ch = chord.Chord(treble_notes)
        treble_ch.duration.quarterLength = duration_qL

        bass_ch = chord.Chord(bass_notes)
        bass_ch.duration.quarterLength = duration_qL

        treble.append(treble_ch)
        bass.append(bass_ch)

    # Barline splitting & ties at measure boundaries
    treble.makeMeasures(inPlace=True)
    bass.makeMeasures(inPlace=True)

    # Add a grand staff brace and finalize notation (adds ties where needed)
    staff_group = layout.StaffGroup([treble, bass], name="Grand Staff", abbreviation="GS", symbol='brace')
    sc.insert(0, staff_group)
    sc.append(treble)
    sc.append(bass)
    sc.makeNotation(inPlace=True)

    sc.write('musicxml', fp=out_musicxml_path)
    sc.write('midi', fp=out_midi_path)

def batch_generate_chorales(input_folder, output_folder=None, **kwargs):
    """
    Iterates over all .txt files in input_folder, converts each to .musicxml and .mid.
    kwargs are passed to chorale_to_grand_staff_musicxml (e.g., merge_identical_runs=True).
    """
    if output_folder is None:
        output_folder = input_folder
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if not filename.lower().endswith(".txt"): #-------------------CHORALE NUMBER-------------------
            continue

        input_path = os.path.join(input_folder, filename)
        chorale_name = os.path.splitext(filename)[0]

        out_musicxml = os.path.join(output_folder, f"{chorale_name}.musicxml")
        out_midi     = os.path.join(output_folder, f"{chorale_name}.mid")

        print(f"Processing {filename} ...")
        chords_list = read_chords_from_txt(input_path)
        if not chords_list:
            print(f"  ⚠️  Skipped (no valid data): {filename}")
            continue

        chorale_to_grand_staff_musicxml(
            chords_list,
            out_musicxml_path=out_musicxml,
            out_midi_path=out_midi,
            **kwargs
        )
        print(f"  ✅ Saved {out_musicxml} and {out_midi}")

if __name__ == "__main__":
    # === USER CONFIGURATION ===
    input_folder = "/content/drive/MyDrive/Master_Thesis/Analysis_Generations/Conf1_Generations_text"
    output_folder = "/content/drive/MyDrive/Master_Thesis/Analysis_Generations/Conf1_scores"

    batch_generate_chorales(
        input_folder,
        output_folder=output_folder,
        time_signature="4/4",
        tempo_qpm=84,
        sixteenth_qL=0.25,
        assume_order_SATB=True,
        merge_identical_runs=True,  # <-- connect identical consecutive chords
        equal_mode="exact"          # or "pitchset"
    )
